# OsteoLab — 2. Entrenamiento

Segundo cuaderno de la serie. En el anterior (`pycon26_01_preparacion_datos.ipynb`) llegamos a
tener una tabla de números (una fila por imagen, columnas con medidas geométricas) y etiquetas de
grupo morfológico. Aquí respondemos a la siguiente pregunta: **¿cómo convertimos esa tabla en un
modelo que generaliza?**

Vamos a entrenar dos familias de modelo distintas sobre los mismos datos, porque resuelven el
problema con estrategias muy diferentes:

1. Un **Random Forest** sobre las features geométricas — un modelo clásico, rápido, interpretable.
2. Una **CNN** (red neuronal convolucional) que mira directamente el recorte de la imagen, en
   varias variantes, incluida una que combina imagen + geometría a la vez.

Y, más importante que entrenar: cómo se **evalúa de forma honesta**, que es donde la mayoría de
proyectos de ML se engañan a sí mismos sin darse cuenta.

## 0. Arrancamos (igual que en el cuaderno 1)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root():
    # Sube desde el directorio actual hasta encontrar params.yaml (raíz del repo).
    start = Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "params.yaml").exists() and (candidate / "training").is_dir():
            return candidate
    raise RuntimeError("No encuentro la raíz del repo. Abre el notebook desde dentro de él.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PROCESSED = REPO_ROOT / "data/processed"
MODELS = REPO_ROOT / "models"

print("Raíz del proyecto:", REPO_ROOT)

In [ ]:
def run(cmd, enabled, cwd=REPO_ROOT):
    # Lanza un script real de entrenamiento y muestra su salida.
    # Entrenar de verdad tarda minutos (Random Forest) o decenas de minutos con GPU (CNN),
    # así que en la charla las celdas están "apagadas" por defecto (enabled=False) y solo
    # imprimen el comando. Para reproducirlo, activa los interruptores de la celda siguiente.
    printable = " ".join(str(c) for c in cmd)
    if not enabled:
        print(f"[celda desactivada, el comando sería]\n$ {printable}")
        return None
    print(f"$ {printable}\n")
    proc = subprocess.run([sys.executable, *cmd[1:]] if cmd[0] == "python" else cmd,
                          cwd=cwd, text=True, capture_output=True)
    print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
    return proc


# Interruptores: en True, cada celda lanza el entrenamiento/evaluación real.
RUN_TRAIN_GEOMETRIC = False   # Random Forest: minutos
RUN_TRAIN_CNN = False         # CNN/híbrido: decenas de minutos con GPU
RUN_EVALUATE = False          # evaluación sobre los bancos externos

## 1. El modelo geométrico: Random Forest

### ¿Qué es un Random Forest, para quien no lo haya usado nunca?

Es un conjunto ("forest" = bosque) de muchos árboles de decisión simples, cada uno entrenado con
una muestra algo distinta de los datos. Un árbol de decisión solo hace preguntas tipo "¿elongation
> 0.7?" y, según la respuesta, sigue por una rama u otra, hasta llegar a una predicción. Un solo
árbol se sobreajusta fácilmente (memoriza en vez de aprender); promediar la predicción de cientos
de árboles distintos corrige eso. Es un modelo clásico — no hay redes neuronales aquí — pero
rápido de entrenar y fácil de interpretar: podemos preguntarle qué features usó más.

Este modelo entrena directamente sobre las columnas numéricas que vimos en el cuaderno 1
(`elongation`, `aspect_ratio`, `hu_moments`, etc.), según la configuración de `params.yaml`.

In [ ]:
import yaml

# params.yaml centraliza los hiperparámetros de ambos modelos (geométrico y CNN).
config = yaml.safe_load((REPO_ROOT / "params.yaml").read_text())
print(config.get("geometric_training"))
print(config.get("geometric_model"))

### Un detalle técnico que importa más de lo que parece: el orden de las columnas

Un Random Forest no sabe que la tercera columna se llama `elongation` — solo sabe que es "la
columna número 3". Si al entrenar el orden de columnas fue A, B, C, y al usar el modelo en
producción se le pasan en orden B, A, C, el modelo seguirá dando una respuesta... pero sin
sentido, porque está leyendo cada medida como si fuera otra. Por eso el entrenamiento guarda aparte
el orden exacto de columnas usado (`geometric_feature_columns_bone_group.joblib`), y la inferencia
lo reutiliza siempre, nunca confía en que las columnas del CSV de turno vengan en el mismo orden.

### Dos evaluaciones, no una

El script de entrenamiento evalúa el modelo **dos veces**, y es importante entender la diferencia:

1. **Split aleatorio por fila** — se parte el CSV al azar en train/test. Es optimista: como cada
   espécimen tiene 24 vistas, es casi seguro que vistas del mismo objeto caigan una en train y
   otra en test, así que el modelo "reconoce" el objeto, no la forma general.
2. **`GroupKFold` repetido 10 veces, con especímenes completos fuera** — esta es la honesta. Se
   garantiza que ningún espécimen aparece a la vez en train y test, así que la métrica mide de
   verdad si el modelo generaliza a huesos que nunca vio.

Ambas métricas se imprimen y se registran en MLflow, precisamente para que nunca se pueda
confundir una con otra por descuido.

In [ ]:
# Entrenamiento real del Random Forest (desactivado por defecto, ver celda de interruptores).
run(["python", "-m", "training.train_geometric"], RUN_TRAIN_GEOMETRIC)
# Con --target bone se podría entrenar sobre los 21 huesos finos en vez de los 9 grupos —
# se probó como comparativa, pero no es lo que corre en producción (ver cuaderno 1, sección 2).

In [ ]:
# El código de los splits "honestos" vive aparte y tiene tests propios — lo imprimimos para verlo.
import inspect

from training import group_splits

print(inspect.getsource(group_splits.subject_ids))

In [ ]:
import joblib
import pandas as pd

# Si el modelo ya está entrenado en esta copia del repo, miramos qué features usó más.
model_files = sorted(p.name for p in MODELS.glob("geometric_*"))
print(model_files)

ruta_modelo = MODELS / "geometric_model_bone_group.joblib"
if ruta_modelo.exists():
    rf = joblib.load(ruta_modelo)
    columnas = joblib.load(MODELS / "geometric_feature_columns_bone_group.joblib")
    importancia = pd.Series(rf.feature_importances_, index=columnas).sort_values(ascending=False)
    print(importancia.head(15).to_string())

### ¿Y si una feature no aportara nada?

`feature_importances_` tiene un sesgo conocido: infla la importancia de features con muchos
valores distintos o correlacionadas entre sí, aunque no aporten información nueva. Una forma más
fiable de comprobar si una columna hace falta de verdad es **permutation importance**: se baraja
al azar una sola columna (rompiendo su relación con la etiqueta) y se mide cuánto cae el accuracy.
Si apenas cae, esa feature no estaba aportando — es candidata a quitar del dataset.


In [ ]:
from sklearn.inspection import permutation_importance

# Reutiliza el mismo dataset combinado del cuaderno 1, con las columnas en el orden que
# espera el modelo (ver la nota de más arriba sobre por qué el orden importa).
dataset_path = PROCESSED / "bone_geometric_features_combined_lab.csv"
if ruta_modelo.exists() and dataset_path.exists():
    datos = pd.read_csv(dataset_path).dropna(subset=columnas)
    X_eval = datos[columnas]
    y_eval = datos["bone_group"]

    resultado = permutation_importance(rf, X_eval, y_eval, n_repeats=10, random_state=0, n_jobs=-1)
    perm_imp = pd.Series(resultado.importances_mean, index=columnas).sort_values(ascending=False)
    print("Caída media de accuracy al barajar cada columna (permutation importance):")
    print(perm_imp.head(15).to_string())
else:
    print("Necesita el modelo entrenado y el dataset combinado — entrena primero (RUN_TRAIN_GEOMETRIC).")

## 2. El modelo CNN / híbrido

### ¿Qué añade una CNN que el Random Forest no tiene?

El Random Forest solo ve los números que ya calculamos con OpenCV — nunca ve la imagen en sí. Una
**CNN** (red neuronal convolucional) mira directamente el recorte de píxeles y aprende sus propias
features visuales, cosas que no se nos ocurrió medir a mano (texturas, patrones locales, relaciones
espaciales complejas). El coste: necesita muchos más datos y mucho más cómputo para entrenar.

Este proyecto no entrena la CNN desde cero: usa **transferencia de aprendizaje**. Parte de un
backbone (`resnet18`, de la librería `timm`) ya entrenado sobre millones de fotos genéricas de
ImageNet, y solo le enseña a adaptar lo que ya sabe de "ver imágenes en general" a "distinguir
grupos de huesos".

Hay tres variantes posibles, según qué entrada recibe la red:

| `--model-type` | Qué entra en la red | Idea |
|---|---|---|
| `cnn_only` | solo el recorte de imagen | backbone + una cabeza (capa final) nueva |
| `hybrid` | recorte + el vector de features geométricas del cuaderno 1 | **la que está en producción hoy** (`hybrid_lab`) |
| `gated_hybrid` | recorte + geometría, con una "puerta" aprendida | la red decide, por imagen, cuánto pesar cada rama |

La variante `hybrid` gana porque combina lo mejor de los dos mundos: la CNN aporta lo visual, y
las medidas geométricas aportan una señal ya probada y estable (como `elongation`) que no depende
de que la red la "descubra" por sí sola con pocos datos.

### Cómo se entrena: dos fases, no una

```
Fase 1 — "cabeza congelada" (--epochs-frozen 5):
    El backbone (resnet18 preentrenado) se queda fijo.
    Solo se entrena la cabeza nueva, con lr=1e-3 (tasa de aprendizaje alta: parte de cero).

Fase 2 — "fine-tuning" (--epochs-finetune 10):
    Se descongela también el backbone, pero con una tasa de aprendizaje mucho más baja
    (--lr-backbone 1e-5): ya sabe ver imágenes en general, solo hay que afinarlo un poco,
    no destrozar lo que ya aprendió de ImageNet con un paso demasiado grande.
```

Dos decisiones más, pensadas específicamente para este problema:

- **`--real-photo-boost 4.0`**: en la pérdida de entrenamiento, cada foto real cuenta como si
  fueran 4 ejemplos. Hay muchísimas más imágenes sintéticas que reales, y sin este ajuste el
  modelo casi ignoraría las fotos reales — que son, precisamente, el dominio que importa.
- **`--select-by-real-bench`**: de todas las épocas de entrenamiento, se elige como modelo final
  la que mejor puntúa en el banco de **fotos reales**, no la que mejor puntúa en validación
  sintética. Son métricas que pueden discrepar, y la que de verdad cuenta es la segunda.

In [ ]:
# Entrenamiento real de la variante en producción (desactivado por defecto).
run(["python", "-m", "training.train_cnn",
     "--model-type", "hybrid",
     "--dataset-path", "data/processed/bone_geometric_features_combined_lab.csv",
     "--run-name", "hybrid_lab",
     "--epochs-frozen", "5", "--epochs-finetune", "10",
     "--real-photo-boost", "4.0",
     "--select-by-real-bench"], RUN_TRAIN_CNN)

### Por qué entrenar tres veces con semillas distintas, no una sola vez

Con un dataset de este tamaño (no son millones de imágenes), una sola corrida de entrenamiento no
te dice si una mejora es real o es simplemente suerte en la inicialización aleatoria de la red.
La regla que se sigue en este proyecto: un cambio solo se acepta como mejora si se mantiene
**en 3 semillas distintas** (`--model-seed 0, 1, 2`). Así se confirmó, por ejemplo, que la receta
`hybrid_lab` era mejor que las alternativas: 0.8701 de accuracy en el banco sintético y 0.8323 en
fotos reales, de forma consistente en las tres corridas — no un pico aislado.

**Nota (2026-10-30)**: esta misma disciplina evitó promover un cambio malo. Se probó quitar de
`combined_lab` las dos variantes de render que repetían especímenes de `renders_lab` (ver cuaderno
1, sección 2) — parecía una limpieza obvia — y, al medirlo de verdad contra el banco grande de
fotos reales, el resultado fue peor (0.8120 → 0.6923 de accuracy, -12 puntos). Se revirtió. El
dataset de producción sigue siendo el de 2938 filas con las tres variantes sintéticas.

## 2.1 Ablación: ¿aporta algo el vector geométrico?

En el cuaderno 1 vimos que las features geométricas, solas, ya superan con holgura a un
baseline tonto. Pero la pregunta que importa de verdad para `hybrid_lab` es otra: **¿la CNN
mejora al recibir también el vector geométrico, o le daría igual no tenerlo?**

Un ablation study es justo esto: entrenar dos modelos idénticos en todo excepto en una pieza
(aquí, si entra o no el vector geométrico), con el mismo dataset, el mismo backbone, las mismas
épocas y la misma semilla — y comparar. Si `hybrid` no le gana a `cnn_only` de forma consistente,
no hay motivo para mantener la complejidad añadida del vector geométrico en producción.


In [ ]:
# Mismo dataset, mismas épocas, misma semilla que hybrid_lab_seed0 — la única diferencia es
# --model-type cnn_only, que no recibe el vector geométrico en la cabeza de la red.
run(["python", "-m", "training.train_cnn",
     "--model-type", "cnn_only",
     "--dataset-path", "data/processed/bone_geometric_features_combined_lab.csv",
     "--run-name", "cnn_only_lab_ablation",
     "--epochs-frozen", "5", "--epochs-finetune", "10",
     "--real-photo-boost", "4.0",
     "--model-seed", "0",
     "--select-by-real-bench"], RUN_TRAIN_CNN)

In [ ]:
import json

# Comparamos las métricas que cada entrenamiento deja escritas en su meta.json, si ambas existen.
metricas_interesantes = ["best_val_accuracy", "best_val_accuracy_real_photos", "accuracy_clean_benchmark"]
filas = {}
for etiqueta, nombre_run in [("cnn_only (sin geometría)", "cnn_only_lab_ablation"),
                              ("hybrid_lab_seed0 (con geometría)", "hybrid_lab_seed0")]:
    ruta_meta = MODELS / f"cnn_training_meta_{nombre_run}_bone_group.json"
    if ruta_meta.exists():
        meta = json.loads(ruta_meta.read_text())
        filas[etiqueta] = {m: meta.get(m) for m in metricas_interesantes}

if filas:
    pd.DataFrame(filas).T
else:
    print("Ninguno de los dos modelos está entrenado todavía en esta copia del repo — "
          "activa RUN_TRAIN_CNN para generarlos, o usa `training.evaluate_cnn` sobre modelos "
          "ya existentes con nombres distintos.")

### Resultado actualizado (2026-10-30, dataset `combined_lab` anterior a la limpieza de duplicados)

Entrenamos `cnn_only_lab_ablation` y `hybrid_lab_seed0` con receta idéntica (mismo dataset, misma
semilla) y los evaluamos en **banco sintético fijo** y en el **banco de fotos reales grande**
(`real_photos_accepted`, split `test`, 117 imágenes — mucho más representativo que las 54 fotos
del banco antiguo citadas abajo):

| | banco sintético fijo | fotos reales (n=117) | balanced accuracy (fotos reales) |
|---|---|---|---|
| `cnn_only` (sin geometría) | 0.4815 | 0.7094 | 0.6755 |
| `hybrid` (con geometría) | 0.5972 | 0.8120 | 0.8098 |

**+11.6 puntos en sintético, +10.3 en fotos reales, +13.4 en balanced accuracy.** Esta vez la
mejora en fotos reales es clara y consistente, no un empate casual — con 117 fotos (el doble que
el banco de 54 citado en la sesión anterior) la señal ya no es ruido de muestra pequeña:

> **Conclusión: el vector geométrico aporta de forma demostrable, tanto en sintético como en
> fotos reales.** No es "apenas ayuda" — es la diferencia entre 0.71 y 0.81 de accuracy en el
> dominio que de verdad importa.

### Lo que ya se había observado antes (referencia histórica, banco de 54 fotos)

Un ablation anterior (`base_datos_osea/HISTORIAL.md`), con una receta algo distinta y un banco de
fotos reales mucho más pequeño (54 fotos): sintético `cnn_only` 0.7685 vs. `hybrid` 0.8241 (mejora
clara); en fotos reales ambos empataron en el total agregado, con desglose por grupo mixto — en su
momento se leyó como "dirección correcta pero banco insuficiente para confirmar". El resultado de
arriba, con más del doble de fotos reales, ya confirma esa dirección con margen.

In [ ]:
for seed in (0, 1, 2):
    run(["python", "-m", "training.train_cnn",
         "--model-type", "hybrid",
         "--dataset-path", "data/processed/bone_geometric_features_combined_lab.csv",
         "--run-name", f"hybrid_lab_seed{seed}",
         "--model-seed", str(seed),
         "--select-by-real-bench"], RUN_TRAIN_CNN)

In [ ]:
# ¿Qué modelos CNN "hybrid" existen ya en esta copia del repo?
print(sorted(p.name for p in MODELS.glob("cnn_*hybrid*")))

## 3. Evaluación honesta: la parte que de verdad importa

Un modelo se evalúa contra **tres bancos de prueba**, ordenados de más fácil a más difícil. Solo
el último es el resultado real del proyecto — los otros dos sirven para depurar, no para
presumir:

1. **Validación interna** (split por filas, dentro del propio entrenamiento) — útil solo para
   detectar que algo está claramente roto (el modelo no aprende nada en absoluto).
2. **Banco sintético excluido** (`bone_geometric_benchmark.csv`) — especímenes completos que el
   modelo nunca vio durante el entrenamiento, pero siguen siendo renders sintéticos.
3. **Banco de fotos reales** — esto ya no es "otro objeto dentro del mismo dominio", es **otro
   dominio entero**. Es la prueba que de verdad dice si el proyecto sirve para algo fuera del
   laboratorio.

Si un cambio mejora el banco 2 pero empeora el banco 3, es una mala señal — probablemente el
modelo se está ajustando a particularidades del render sintético, no a la forma real del hueso.

In [ ]:
run(["python", "-m", "training.evaluate"], RUN_EVALUATE)                                  # geométrico
run(["python", "-m", "training.evaluate_cnn", "--model-type", "hybrid"], RUN_EVALUATE)    # CNN/híbrido
run(["python", "-m", "training.evaluate_real_photos", "--segmenter", "grounded_sam"], RUN_EVALUATE)
run(["python", "-m", "training.evaluate_cnn_real_photos",
     "--model-type", "hybrid", "--segmenter", "grounded_sam", "--source", "benchmark"], RUN_EVALUATE)

In [ ]:
# Informes que estas evaluaciones ya dejaron escritos en disco (matrices de confusión, ROC...).
for p in sorted((REPO_ROOT / "reports").rglob("*")):
    if p.is_file():
        print(p.relative_to(REPO_ROOT))

In [ ]:
run(["python", "-m", "training.plot_roc_curves"], RUN_EVALUATE)   # curvas ROC del modelo geométrico
run(["python", "-m", "training.plot_roc_cnn"], RUN_EVALUATE)      # curvas ROC de la CNN/híbrido

### Lo que dicen estos resultados

El número agregado (accuracy global) no basta — hay que mirar clase por clase. En este proyecto,
`hueso_plano` sigue siendo la clase más débil: tiene pocos especímenes y su forma, una vez
segmentada, se parece a `cranio` y a `pelvis`.

Un ejemplo muy concreto de cómo mover la métrica: en un piloto más fino (clasificar también el
taxón — humano, carnívoro, ungulado...) dentro del grupo `cranio`, el recall de `ungulado` pasó de
**0.0 a 0.6** con un solo cambio: añadir fotos reales de ese grupo concreto. Con tan pocas clases
por rama, un puñado de fotos reales del dominio correcto mueve más la métrica que cualquier ajuste
de arquitectura de red.

## Lo que viene después

Ya tenemos modelos entrenados y evaluados con honestidad. El siguiente cuaderno
(`pycon26_03_inferencia.ipynb`) los pone a trabajar juntos: una regla rápida antes de la CNN
("cascada"), un segundo nivel que afina hueso concreto o taxón, y la API que expone todo esto a
una aplicación real.